In [5]:
# -*- coding: utf-8 -*-
"""
Application of Generalized Linear Models (GLMs) for Dengue Forecasting in Bangladesh
-----------------------------------------------------------------------------------
ONE script:  data -> models -> figures -> poster (.pptx)

Install once:
    pip install pandas numpy matplotlib seaborn scipy statsmodels scikit-learn patsy openpyxl python-pptx

Run:
    python dengue_poster.py
Output:
    Dengue_Poster.pptx   (+ all figures in ./poster_figs)
"""
import os
import re
import math
import warnings

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from pptx import Presentation
from pptx.util import Inches, Pt
from pptx.dml.color import RGBColor
from pptx.enum.text import PP_ALIGN, MSO_ANCHOR
from pptx.enum.shapes import MSO_SHAPE
from pptx.oxml.ns import qn
from lxml import etree

warnings.filterwarnings("ignore")

# =============================================================================
# 1. CONFIG  (edit this part)
# =============================================================================
DATA_FILE = r"C:\Users\HP\Desktop\STAT 3103\Dengue data.xlsx"         # put the Excel file next to this script
SHEET = "Data"
OUT_PPTX = "Dengue_Poster.pptx"
FIG_DIR = "poster_figs"

POSTER_W, POSTER_H = 36.0, 42.0         # inches (change if your teacher gives a size)

TITLE = "Application of Generalized Linear Models (GLMs) for Dengue Forecasting in Bangladesh"
AUTHORS = "Your Name¹, Team Member 2¹, Team Member 3¹"            # <-- EDIT
AFFILIATION = "¹Department of Statistics, Begum Rokeya University, Rangpur-5400, Bangladesh"  # <-- EDIT
CONTACT = ["Your Name", "Department of Statistics, Begum Rokeya University, Rangpur, Bangladesh",
           "Email: yourmail@gmail.com   |   Phone: 01XXXXXXXXX"]  # <-- EDIT
FOOTER_RIGHT = "Department of Statistics  •  Course: STAT 3103"  # <-- EDIT

TRAIN_FRACTION = 0.80
NUM = ["T2M", "RH2M", "PRECTOTCORR", "Pd"]
RHS = "trend + C(Months) + T2M_z + RH2M_z + PRECTOTCORR_z + Pd_z"
FORMULA = "Cases ~ " + RHS
MODEL_ORDER = ["Poisson", "NB", "ZIP", "ZINB"]
MONTHS = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
LABELS = {"trend": "Trend (per year)", "T2M_z": "Temperature (per SD)",
          "RH2M_z": "Humidity (per SD)", "PRECTOTCORR_z": "Rainfall (per SD)",
          "Pd_z": "Population density (per SD)"}
COV_KEYS = list(LABELS.keys())

# ---- poster look ----
FONT = "Times New Roman"
NAVY = RGBColor(0x0B, 0x2A, 0x4A)
MID = RGBColor(0x1F, 0x4E, 0x79)
LIGHT = RGBColor(0xE8, 0xEF, 0xF7)
DARK = RGBColor(0x11, 0x11, 0x11)
WHITE = RGBColor(0xFF, 0xFF, 0xFF)
MODEL_COL = {"Poisson": "#d95f02", "NB": "#1b9e77", "ZIP": "#7570b3", "ZINB": "#e7298a"}

BODY_PT = 25
CAP_PT = 20
HDR_PT = 34
HDR_H = 1.0
CHAR_W = 0.46     # avg glyph width (em) used for text-height estimation
LINE_H = 1.17

plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "Liberation Serif", "DejaVu Serif"],
    "axes.spines.top": False, "axes.spines.right": False,
})


# =============================================================================
# 2. DATA + MODELS  (same logic as your notebook)
# =============================================================================
def load_data():
    df = pd.read_excel(DATA_FILE, sheet_name=SHEET)
    df.columns = df.columns.str.strip()
    required = ["Cases", "Year", "Months"] + NUM
    missing = set(required) - set(df.columns)
    if missing:
        raise ValueError(f"Missing columns: {missing}")
    df["Date"] = pd.to_datetime(dict(year=df["Year"], month=df["Months"], day=1), errors="coerce")
    df = df.sort_values("Date").reset_index(drop=True)
    df = df.dropna(subset=required + ["Date"]).copy()
    df["Cases"] = df["Cases"].astype(int)
    if df["Date"].duplicated().any():
        raise ValueError("Resolve duplicate months before modeling.")
    if (df["Cases"] < 0).any():
        raise ValueError("Negative count detected.")
    df["trend"] = (df["Date"] - df["Date"].min()).dt.days / 365.25
    return df.reset_index(drop=True)


def add_z(df, ref):
    """z-score continuous predictors (keeps optimisation stable; IRR is per 1 SD)."""
    df = df.copy()
    for c in NUM:
        df[c + "_z"] = (df[c] - ref[c].mean()) / ref[c].std()
    return df


def robust_fit(make_model, methods=("bfgs", "nm"), maxiter=2000):
    """Try several optimisers until one converges."""
    last = None
    for m in methods:
        try:
            model = make_model()
            res = model.fit(method=m, maxiter=maxiter, disp=0)
            if m == "nm":                         # polish Nelder-Mead with BFGS
                try:
                    res2 = make_model().fit(start_params=res.params, method="bfgs",
                                            maxiter=maxiter, disp=0)
                    res = res2
                except Exception:
                    pass
            last = res
            conv = getattr(res, "mle_retvals", {}).get("converged", True)
            if conv:
                return res
        except Exception as e:
            print(f"   (optimiser {m} failed: {e})")
    if last is None:
        raise RuntimeError("Model could not be fitted.")
    print("   WARNING: model did not fully converge - interpret with care.")
    return last


def run_analysis():
    import statsmodels.api as sm
    import statsmodels.formula.api as smf
    from patsy import dmatrix, build_design_matrices
    from scipy.stats import nbinom
    from statsmodels.discrete.count_model import ZeroInflatedPoisson, ZeroInflatedNegativeBinomialP
    from statsmodels.stats.outliers_influence import variance_inflation_factor
    from sklearn.metrics import mean_absolute_error, mean_squared_error

    data = load_data()
    n = len(data)
    split = int(n * TRAIN_FRACTION)
    train, test = data.iloc[:split].copy(), data.iloc[split:].copy()
    tz, sz, fz = add_z(train, train), add_z(test, train), add_z(data, data)

    # ---------- descriptive / diagnostics ----------
    mean, var = data["Cases"].mean(), data["Cases"].var()
    zeros = int((data["Cases"] == 0).sum())
    Xv = sm.add_constant(data[NUM])
    vif = [variance_inflation_factor(Xv.values, i) for i in range(1, Xv.shape[1])]
    corr_trend_pd = float(np.corrcoef(data["trend"], data["Pd"])[0, 1])

    # ---------- fit the four models on TRAIN ----------
    print("Fitting Poisson ...")
    pois = smf.glm(FORMULA, tz, family=sm.families.Poisson()).fit()
    print("Fitting Negative Binomial ...")
    nb = robust_fit(lambda: smf.negativebinomial(FORMULA, tz), methods=("newton", "bfgs", "nm"))

    X_tr = dmatrix("1 + " + RHS, tz, return_type="dataframe")
    X_te = build_design_matrices([X_tr.design_info], sz, return_type="dataframe")[0]
    Z_tr = dmatrix("1 + trend", tz, return_type="dataframe")
    Z_te = build_design_matrices([Z_tr.design_info], sz, return_type="dataframe")[0]
    y_tr = tz["Cases"]

    print("Fitting ZIP ...")
    zip_m = robust_fit(lambda: ZeroInflatedPoisson(endog=y_tr, exog=X_tr, exog_infl=Z_tr,
                                                   inflation="logit"))
    print("Fitting ZINB ...")
    zinb_m = robust_fit(lambda: ZeroInflatedNegativeBinomialP(endog=y_tr, exog=X_tr, exog_infl=Z_tr,
                                                              inflation="logit", p=2))
    models = {"Poisson": pois, "NB": nb, "ZIP": zip_m, "ZINB": zinb_m}

    # ---------- predictions on TEST ----------
    preds = {
        "Poisson": pois.predict(sz),
        "NB": nb.predict(sz),
        "ZIP": zip_m.predict(exog=X_te, exog_infl=Z_te),
        "ZINB": zinb_m.predict(exog=X_te, exog_infl=Z_te),
    }
    y_test = test["Cases"].values.astype(float)
    for k in preds:
        p = np.nan_to_num(np.asarray(preds[k], dtype=float).ravel(), nan=0.0, posinf=1e10, neginf=0.0)
        preds[k] = np.clip(p, 0, 1e12)

    # ---------- Pearson dispersion for each model (train) ----------
    ytr = y_tr.values.astype(float)

    def pearson(name):
        try:
            m = models[name]
            k = len(m.params)
            if name == "Poisson":
                mu = np.asarray(m.predict(tz)); mean_, v = mu, mu
            elif name == "NB":
                mu = np.asarray(m.predict(tz)); a = float(np.asarray(m.params)[-1])
                mean_, v = mu, mu + a * mu ** 2
            else:
                mu = np.asarray(m.predict(exog=X_tr, exog_infl=Z_tr, which="mean-main"))
                pm = np.asarray(m.predict(exog=X_tr, exog_infl=Z_tr, which="prob-main"))
                pi = 1 - pm
                mean_ = pm * mu
                if name == "ZIP":
                    v = pm * mu * (1 + pi * mu)
                else:
                    a = float(np.asarray(m.params)[-1])
                    v = pm * mu * (1 + a * mu + pi * mu)
            return float(np.sum((ytr - mean_) ** 2 / v) / (len(ytr) - k))
        except Exception:
            return np.nan

    rows = []
    for name in MODEL_ORDER:
        m = models[name]
        k = len(m.params)
        aic = -2 * m.llf + 2 * k
        bic = -2 * m.llf + k * np.log(len(train))
        pr = preds[name]
        rows.append({"Model": name, "AIC": aic, "BIC": bic, "k": k,
                     "Dispersion": pearson(name),
                     "MAE": mean_absolute_error(y_test, pr),
                     "RMSE": float(np.sqrt(mean_squared_error(y_test, pr)))})
    comp = pd.DataFrame(rows).set_index("Model")
    comp["dAIC"] = comp["AIC"] - comp["AIC"].min()

    # ---------- expected zeros (train) ----------
    mu_p = np.asarray(pois.predict(tz))
    a_nb = float(np.asarray(nb.params)[-1])
    mu_n = np.asarray(nb.predict(tz))
    size = 1.0 / a_nb
    exp0_p = float(np.exp(-mu_p).sum())
    exp0_n = float(nbinom.pmf(0, size, size / (size + mu_n)).sum())
    obs0_tr = int((train["Cases"] == 0).sum())

    # ---------- final NB on all data -> IRR ----------
    print("Fitting final NB on all data ...")
    final = robust_fit(lambda: smf.negativebinomial(FORMULA, fz), methods=("newton", "bfgs", "nm"))
    params, ci, pv = final.params, final.conf_int(), final.pvalues
    recs = []
    for name in params.index:
        if name in ("Intercept", "alpha"):
            continue
        if name in LABELS:
            lab = LABELS[name]
        else:
            mm = re.search(r"\[T\.(\d+)\]", name)
            lab = f"Month: {MONTHS[int(mm.group(1)) - 1]} vs Jan" if mm else name
        recs.append({"name": name, "label": lab, "IRR": float(np.exp(params[name])),
                     "lo": float(np.exp(ci.loc[name, 0])), "hi": float(np.exp(ci.loc[name, 1])),
                     "p": float(pv[name])})
    irr = pd.DataFrame(recs)

    return build_result_dict(data, train, test, comp, preds, irr, mean, var, zeros, vif,
                             corr_trend_pd, exp0_p, exp0_n, obs0_tr)


def build_result_dict(data, train, test, comp, preds, irr, mean, var, zeros, vif,
                      corr_trend_pd, exp0_p, exp0_n, obs0_tr):
    fmt = lambda d: d.strftime("%b %Y")
    mm = data.groupby("Months")["Cases"].mean()
    tmax = test.loc[test["Cases"].idxmax()]
    return dict(
        data=data, train=train, test=test, comp=comp, preds=preds, irr=irr,
        n=len(data), n_train=len(train), n_test=len(test),
        period=f"{fmt(data['Date'].min())} – {fmt(data['Date'].max())}",
        train_period=f"{fmt(train['Date'].min())} – {fmt(train['Date'].max())}",
        test_period=f"{fmt(test['Date'].min())} – {fmt(test['Date'].max())}",
        mean=mean, var=var, var_mean=var / mean, zeros=zeros, zero_pct=100 * zeros / len(data),
        vif_max=max(vif), corr_trend_pd=corr_trend_pd,
        exp0_p=exp0_p, exp0_n=exp0_n, obs0_tr=obs0_tr,
        best_fit=comp["AIC"].idxmin(), best_bic=comp["BIC"].idxmin(),
        best_fc=comp["MAE"].idxmin(), best_rmse=comp["RMSE"].idxmin(),
        peak_month=MONTHS[int(mm.idxmax()) - 1], peak_month_mean=float(mm.max()),
        test_max=int(tmax["Cases"]), test_max_date=fmt(tmax["Date"]),
        total_cases=int(data["Cases"].sum()),
        year_max=int(data.groupby("Year")["Cases"].sum().idxmax()),
        year_max_cases=int(data.groupby("Year")["Cases"].sum().max()),
    )


# =============================================================================
# 3. FIGURES  (drawn at the exact size they will have on the poster)
# =============================================================================
def _style(ax, fs=13):
    ax.tick_params(labelsize=fs)
    ax.grid(alpha=0.25)


def fig_eda(R, path, w, h):
    d, tr = R["data"], R["train"]
    fig = plt.figure(figsize=(w, h))
    gs = fig.add_gridspec(1, 2, width_ratios=[1.7, 1], wspace=0.28)
    ax = fig.add_subplot(gs[0])
    ax.plot(d["Date"], d["Cases"], color="#0B2A4A", lw=1.6)
    ax.axvspan(tr["Date"].iloc[-1], d["Date"].iloc[-1], color="#f4a261", alpha=0.25, label="Test period")
    ax.set_title("(a) Monthly dengue cases", fontsize=15, fontweight="bold")
    ax.set_ylabel("Cases", fontsize=14)
    ax.legend(fontsize=12, frameon=False, loc="upper left")
    _style(ax)
    ax2 = fig.add_subplot(gs[1])
    mm = d.groupby("Months")["Cases"].mean()
    ax2.bar(range(1, 13), mm.values, color="#1F4E79")
    ax2.set_xticks(range(1, 13))
    ax2.set_xticklabels([m[0] for m in MONTHS])
    ax2.set_title("(b) Mean cases by month", fontsize=15, fontweight="bold")
    _style(ax2)
    fig.subplots_adjust(left=0.08, right=0.99, bottom=0.14, top=0.88)
    fig.savefig(path, dpi=200); plt.close(fig)


def fig_forecast(R, path, w, h):
    te = R["test"]
    fig, ax = plt.subplots(figsize=(w, h))
    ax.plot(te["Date"], te["Cases"], color="black", lw=2.4, marker="o", ms=4, label="Observed", zorder=5)
    for k in MODEL_ORDER:
        ax.plot(te["Date"], R["preds"][k], color=MODEL_COL[k], lw=1.6, label=k)
    ax.set_yscale("symlog", linthresh=10)
    ax.set_ylabel("Dengue cases (log scale)", fontsize=14)
    ax.set_title("A   Observed vs predicted cases (test period)", fontsize=15, fontweight="bold", loc="left")
    ax.set_ylim(top=ax.get_ylim()[1] * 8)
    ax.legend(ncol=5, fontsize=12, frameon=False, loc="upper left")
    _style(ax)
    fig.subplots_adjust(left=0.09, right=0.99, bottom=0.12, top=0.88)
    fig.savefig(path, dpi=200); plt.close(fig)


def fig_forest(R, path, w, h):
    irr = R["irr"].iloc[::-1].reset_index(drop=True)
    fig, ax = plt.subplots(figsize=(w, h))
    y = np.arange(len(irr))
    lo = np.clip(irr["lo"], 1e-3, None); hi = np.clip(irr["hi"], None, 1e6)
    col = ["#c0392b" if p < 0.05 else "#7f8c8d" for p in irr["p"]]
    for yi, l, u, c in zip(y, lo, hi, col):
        ax.plot([l, u], [yi, yi], color=c, lw=2)
    ax.scatter(irr["IRR"], y, c=col, s=45, zorder=3)
    ax.axvline(1, color="black", ls="--", lw=1)
    ax.set_xscale("log")
    ax.set_yticks(y); ax.set_yticklabels(irr["label"], fontsize=12)
    ax.set_xlabel("Incidence Rate Ratio (95% CI, log scale)   — red = p < 0.05", fontsize=13)
    ax.set_title("B   Adjusted IRR from the final NB model", fontsize=15, fontweight="bold", loc="left")
    _style(ax, 12)
    fig.subplots_adjust(left=0.27, right=0.98, bottom=0.12, top=0.92)
    fig.savefig(path, dpi=200); plt.close(fig)


def fig_scatter(R, path, w, h):
    y = R["test"]["Cases"].values.astype(float)
    fig, axes = plt.subplots(1, 4, figsize=(w, h), sharex=True, sharey=True)
    lim = [1, max(y.max(), 10) * 3]
    for ax, k in zip(axes, MODEL_ORDER):
        p = np.clip(R["preds"][k], 0, 1e9)
        ax.scatter(y + 1, p + 1, s=16, color=MODEL_COL[k], alpha=0.8)
        ax.plot(lim, lim, color="red", lw=1)
        ax.set_xscale("log"); ax.set_yscale("log")
        ax.set_xlim(lim); ax.set_ylim(lim)
        ax.set_title(k, fontsize=14, fontweight="bold")
        ax.set_xlabel("Observed + 1", fontsize=12)
        _style(ax, 11)
    axes[0].set_ylabel("Predicted + 1", fontsize=12)
    fig.suptitle("C   Goodness of fit: predicted vs observed (log scale)", x=0.01, ha="left",
                 fontsize=15, fontweight="bold")
    fig.subplots_adjust(left=0.07, right=0.99, bottom=0.2, top=0.74, wspace=0.12)
    fig.savefig(path, dpi=200); plt.close(fig)


# =============================================================================
# 4. POSTER TEXT (numbers come from the real results)
# =============================================================================
def num(x, d=0):
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return "–"
    if abs(x) >= 1e7:
        return f"{x:.2e}".replace("e+0", "×10^").replace("e+", "×10^")
    return f"{x:,.{d}f}"


def significant(R):
    irr = R["irr"].set_index("name")
    out = []
    for k in COV_KEYS:
        if k in irr.index and irr.loc[k, "p"] < 0.05:
            r = irr.loc[k]
            out.append((LABELS[k].split(" (")[0], r["IRR"], r["p"]))
    return out


def sig_text(R):
    s = significant(R)
    if not s:
        return "no covariate reached p < 0.05 in the final NB model"
    return "; ".join(f"{n} (IRR = {i:.2f})" for n, i, _ in s)


def make_texts(R):
    c = R["comp"]
    bf, bc = R["best_fit"], R["best_fc"]
    T = {}
    T["abstract"] = [
        ([("Background", True)], False),
        ("Dengue is a mosquito-borne viral disease and a recurring public-health problem in Bangladesh. "
         "Reliable forecasting of monthly cases, together with knowledge of the factors that drive them, "
         "supports early warning and resource planning.", False),
        ([("Methods", True)], False),
        (f"Monthly dengue cases ({R['period']}, n = {R['n']}) with temperature, humidity, rainfall and "
         f"population density were analysed. Poisson, Negative Binomial (NB), Zero-Inflated Poisson (ZIP) and "
         f"Zero-Inflated NB (ZINB) models were fitted on the first {R['n_train']} months and evaluated on the "
         f"last {R['n_test']} months ({R['test_period']}).", False),
        ([("Results", True)], False),
        ([("Best-fitting model: ", True), (f"{bf} (AIC = {num(c.loc[bf, 'AIC'], 1)}, BIC = {num(c.loc[R['best_bic'], 'BIC'], 1)}).", False)], False),
        ([("Best forecasting model: ", True), (f"{bc} (MAE = {num(c.loc[bc, 'MAE'], 0)}, RMSE = {num(c.loc[R['best_rmse'], 'RMSE'], 0)}).", False)], False),
        ([("Significant factors: ", True), (sig_text(R) + ".", False)], False),
        ([("Seasonality: ", True), (f"cases peak in {R['peak_month']} on average.", False)], False),
        ([("Conclusion", True)], False),
        (f"Dengue counts are strongly over-dispersed (variance/mean = {R['var_mean']:,.0f}) and zero-heavy "
         f"({R['zero_pct']:.1f}% zero months), so count GLMs that handle these features are essential for "
         f"dengue forecasting in Bangladesh.", False),
    ]
    T["intro"] = [
        "Dengue is a mosquito-borne viral infection transmitted mainly by Aedes aegypti and Aedes albopictus. "
        "It is a major public-health problem in tropical and sub-tropical countries, and Bangladesh experiences "
        "recurrent outbreaks, mostly in the rainy season.",
        "Monthly dengue counts are non-negative integers, strongly right-skewed, contain many zero months and "
        "show extra-Poisson variation. Ordinary linear regression is therefore unsuitable; Generalized Linear "
        "Models (GLMs) for counts – Poisson, Negative Binomial (NB), Zero-Inflated Poisson (ZIP) and "
        "Zero-Inflated NB (ZINB) – are more appropriate.",
        "Temperature, humidity, rainfall and population density affect mosquito breeding and virus "
        "transmission, so they are natural predictors for an early-warning and forecasting model.",
        ([("Objectives", True)], False),
        ("To forecast monthly dengue cases in Bangladesh using Poisson, NB, ZIP and ZINB models.", True),
        ("To measure the association of climate, population density, time trend and seasonality with dengue "
         "incidence through incidence rate ratios (IRR).", True),
        ("To select the best-fitting and best-forecasting model using AIC, BIC, MAE and RMSE.", True),
    ]
    T["data_bul"] = [
        (f"{R['n']} monthly observations ({R['period']}); {R['total_cases']:,} dengue cases in total.", True),
        (f"Mean = {R['mean']:,.1f} and variance = {R['var']:,.0f} cases/month (variance/mean = {R['var_mean']:,.0f}).", True),
        (f"{R['zeros']} zero-case months ({R['zero_pct']:.1f}%).", True),
        (f"Highest yearly total: {R['year_max']} ({R['year_max_cases']:,} cases).", True),
    ]
    T["fig1_cap"] = ("Figure 1: ", "(a) Monthly dengue cases in Bangladesh; shaded area = test period. "
                                   "(b) Average cases by calendar month.")
    T["methods"] = [
        ([("Data: ", True), ("monthly dengue cases (DGHS), NASA MERRA-2 climate (T2M, RH2M, PRECTOTCORR) and "
                             "World Bank population density (Pd).", False)], True),
        ([("Model: ", True), ("log(μ) = β₀ + β₁·trend + Σ βₘ·Month + β·(T2M, RH2M, PRECTOTCORR, Pd); "
                              "continuous predictors are standardised, IRR = exp(β).", False)], True),
        ([("Checks: ", True), (f"variance/mean, Pearson χ²/df and observed vs expected zeros; "
                               f"max VIF = {R['vif_max']:.2f} for the four covariates.", False)], True),
    ]
    T["fig2_cap"] = ("Figure 2: ", "Study design – from data collection to model selection and interpretation.")
    T["fig3_cap"] = ("Figure 3: ", "A) Observed vs predicted cases (test set); B) adjusted IRR (final NB model on all "
                                   "data); C) predicted vs observed on log scale.")
    T["table_cap"] = ("Table 1: ", "Model selection (training set) and forecast accuracy (test set).")

    sig = significant(R)
    zero_line = (f"Observed zeros in the training data = {R['obs0_tr']}; expected by Poisson = {R['exp0_p']:.1f}, "
                 f"by NB = {R['exp0_n']:.1f}.")
    T["discussion"] = [
        ([(f"{bf} was the best-fitting model", True),
          (f" (lowest AIC = {num(c.loc[bf, 'AIC'], 1)}); BIC favoured {R['best_bic']}.", False)], True),
        ([("Over-dispersion: ", True),
          (f"variance/mean = {R['var_mean']:,.0f}; the Poisson model has Pearson χ²/df = "
           f"{num(c.loc['Poisson', 'Dispersion'], 1)}, so its equal-mean-variance assumption fails.", False)], True),
        ([("Zeros: ", True), (zero_line, False)], True),
        ([(f"{bc} gave the most accurate forecasts", True),
          (f" (MAE = {num(c.loc[bc, 'MAE'], 0)}). The test period contains the largest outbreak "
           f"({R['test_max']:,} cases in {R['test_max_date']}), which makes forecasting difficult.", False)], True),
        ([("Associations: ", True), (sig_text(R) + ".", False)], True),
        ([("Seasonality: ", True), (f"dengue peaks in {R['peak_month']} (mean {R['peak_month_mean']:,.0f} cases), "
                                    f"confirming a strong seasonal pattern.", False)], True),
    ]
    T["conclusion"] = [
        (f"The {bf} model fitted the dengue data best, while {bc} produced the lowest forecast error "
         f"(MAE = {num(c.loc[bc, 'MAE'], 0)}).", True),
        ("Poisson regression is not adequate because of extreme over-dispersion; NB-based models are preferred.", True),
        (f"Time trend, seasonality and climate/population variables explain part of the variation "
         f"({'significant: ' + sig_text(R) if sig else 'no covariate significant'}).", True),
        ("GLM-based forecasts can support early warning, vector-control planning and hospital preparedness.", True),
    ]
    T["limitation"] = [
        ("National monthly data hide district-level and within-month variation.", True),
        (f"Population density and the linear trend are strongly correlated (r = {R['corr_trend_pd']:.2f}), "
         f"so their separate effects are hard to identify.", True),
        ("The test period includes an exceptional outbreak and the COVID-19 period, which may distort forecasts.", True),
        ("No lagged climate effects, spatial effects or vector (Aedes index) data were used.", True),
    ]
    T["refs"] = [
        "1. World Health Organization. Dengue and severe dengue – fact sheet.",
        "2. Dengue case data: Directorate General of Health Services (DGHS), Bangladesh; "
        "https://doi.org/10.1371/journal.pntd.0010130",
        "3. NASA POWER / MERRA-2 meteorological data; World Bank Open Data (population density).",
        "4. Lambert, D. (1992). Zero-inflated Poisson regression. Technometrics 34(1): 1–14.",
        "5. Cameron, A. C. & Trivedi, P. K. (2013). Regression Analysis of Count Data. Cambridge Univ. Press.",
        "6. Hilbe, J. M. (2011). Negative Binomial Regression. Cambridge Univ. Press.",
    ]
    return T


# =============================================================================
# 5. PPTX HELPERS
# =============================================================================
def norm(spec):
    """paragraph spec -> (runs, bullet)"""
    bullet = False
    if isinstance(spec, tuple):
        runs, bullet = spec
    else:
        runs = spec
    if isinstance(runs, str):
        runs = [(runs, False)]
    return runs, bullet


def est_h(paras, size, width, after=6, margin=0.12):
    inner = width - 2 * margin
    total = 0.0
    for spec in paras:
        runs, bullet = norm(spec)
        text = "".join(t for t, _ in runs)
        w = inner - (size * 1.1 / 72 if bullet else 0)
        cpl = max(1, int(w * 72 / (size * CHAR_W)))
        lines = 0
        for seg in text.split("\n"):
            cur, nl = 0, 1
            for word in seg.split(" "):
                L = len(word) + (1 if cur else 0)
                if cur + L > cpl:
                    nl += 1; cur = len(word)
                else:
                    cur += L
            lines += nl
        total += lines * size * LINE_H / 72 + after / 72
    return total + 0.1


def set_bullet(p, size):
    pPr = p._p.get_or_add_pPr()
    ind = int(Pt(size) * 1.1)
    pPr.set("marL", str(ind)); pPr.set("indent", str(-ind))
    f = etree.SubElement(pPr, qn("a:buFont")); f.set("typeface", "Arial")
    b = etree.SubElement(pPr, qn("a:buChar")); b.set("char", "▪")


def add_text(slide, x, y, w, h, paras, size, color=DARK, align=PP_ALIGN.LEFT,
             anchor=MSO_ANCHOR.TOP, after=6, margin=0.12):
    tb = slide.shapes.add_textbox(Inches(x), Inches(y), Inches(w), Inches(h))
    tf = tb.text_frame
    tf.word_wrap = True
    tf.margin_left = tf.margin_right = Inches(margin)
    tf.margin_top = tf.margin_bottom = Inches(0.04)
    tf.vertical_anchor = anchor
    first = True
    for spec in paras:
        runs, bullet = norm(spec)
        p = tf.paragraphs[0] if first else tf.add_paragraph()
        first = False
        p.alignment = align
        p.space_after = Pt(after)
        for t, b in runs:
            r = p.add_run(); r.text = t
            r.font.size = Pt(size); r.font.bold = b; r.font.name = FONT; r.font.color.rgb = color
        if bullet:
            set_bullet(p, size)
    return tb


def add_rect(slide, x, y, w, h, fill=None, line=None, lw=2.0, shape=MSO_SHAPE.RECTANGLE):
    s = slide.shapes.add_shape(shape, Inches(x), Inches(y), Inches(w), Inches(h))
    s.shadow.inherit = False
    if fill is None:
        s.fill.background()
    else:
        s.fill.solid(); s.fill.fore_color.rgb = fill
    if line is None:
        s.line.fill.background()
    else:
        s.line.color.rgb = line; s.line.width = Pt(lw)
    return s


def shape_text(s, text, size, color=WHITE, bold=True, align=PP_ALIGN.CENTER):
    tf = s.text_frame
    tf.word_wrap = True
    tf.margin_left = tf.margin_right = Inches(0.06)
    tf.margin_top = tf.margin_bottom = Inches(0.02)
    tf.vertical_anchor = MSO_ANCHOR.MIDDLE
    lines = text.split("\n")
    for i, ln in enumerate(lines):
        p = tf.paragraphs[0] if i == 0 else tf.add_paragraph()
        p.alignment = align
        r = p.add_run(); r.text = ln
        r.font.size = Pt(size); r.font.name = FONT; r.font.color.rgb = color
        r.font.bold = bold if i == 0 else False


def draw_section(slide, x, y, w, h, title):
    add_rect(slide, x, y, w, h, fill=WHITE, line=NAVY, lw=2.5)
    hdr = add_rect(slide, x, y, w, HDR_H, fill=NAVY)
    shape_text(hdr, title, HDR_PT)


def draw_flow(slide, x, y, w, h, R):
    stages = [
        ("Data collection", [f"Dengue cases\nDGHS, n = {R['n']} months", "Climate\nT2M, RH2M, rainfall (NASA)",
                             "Demography\nPopulation density (WB)"]),
        ("Data preprocessing", ["Quality checks • date index • trend & month factor • VIF"]),
        ("Time-based split", [f"Training set (80%)\n{R['train_period']}", f"Test set (20%)\n{R['test_period']}"]),
        ("Diagnostics", ["Over-dispersion\n(Var/Mean, Pearson χ²/df)", "Excess zeros\n(observed vs expected)"]),
        ("GLM fitting (training set)", ["Poisson", "Negative\nBinomial", "ZIP", "ZINB"]),
        ("Evaluation (test set)", ["AIC / BIC", "Pearson χ²/df", "MAE", "RMSE"]),
        ("Best model", ["Final NB model on all data → IRR interpretation"]),
    ]
    n = len(stages)
    arrow_h = 0.2
    band_h = (h - (n - 1) * arrow_h) / n
    cy = y
    for i, (title, boxes) in enumerate(stages):
        band = add_rect(slide, x, cy, w, band_h, fill=LIGHT, line=MID, lw=1.25, shape=MSO_SHAPE.ROUNDED_RECTANGLE)
        band.adjustments[0] = 0.12
        add_text(slide, x, cy + 0.02, w, 0.34, [([(title, True)], False)], 17, color=NAVY,
                 align=PP_ALIGN.CENTER, after=0, margin=0.05)
        k = len(boxes)
        gap = 0.12
        bw = (w - 0.3 - (k - 1) * gap) / k
        bh = band_h - 0.45
        for j, t in enumerate(boxes):
            b = add_rect(slide, x + 0.15 + j * (bw + gap), cy + 0.38, bw, bh, fill=MID,
                         shape=MSO_SHAPE.ROUNDED_RECTANGLE)
            b.adjustments[0] = 0.15
            shape_text(b, t, 15)
        cy += band_h
        if i < n - 1:
            a = add_rect(slide, x + w / 2 - 0.2, cy + 0.01, 0.4, arrow_h - 0.02, fill=NAVY,
                         shape=MSO_SHAPE.DOWN_ARROW)
            cy += arrow_h


def add_table(slide, x, y, w, R, size=15, row_h=0.55):
    c = R["comp"]
    head = ["Model", "AIC", "BIC", "ΔAIC", "Pearson χ²/df", "MAE", "RMSE"]
    rows = [head]
    for k in MODEL_ORDER:
        r = c.loc[k]
        rows.append([k, num(r["AIC"], 1), num(r["BIC"], 1), num(r["dAIC"], 2),
                     num(r["Dispersion"], 2), num(r["MAE"], 0), num(r["RMSE"], 0)])
    shp = slide.shapes.add_table(len(rows), len(head), Inches(x), Inches(y), Inches(w),
                                 Inches(row_h * len(rows)))
    tbl = shp.table
    widths = [1.2, 1.7, 1.7, 1.3, 1.6, 1.4, 1.4]
    sc = w / sum(widths)
    for i, wd in enumerate(widths):
        tbl.columns[i].width = Inches(wd * sc)
    for i, row in enumerate(rows):
        tbl.rows[i].height = Inches(row_h)
        for j, val in enumerate(row):
            cell = tbl.cell(i, j)
            cell.fill.solid()
            cell.fill.fore_color.rgb = NAVY if i == 0 else (LIGHT if i % 2 == 0 else WHITE)
            cell.vertical_anchor = MSO_ANCHOR.MIDDLE
            cell.margin_left = cell.margin_right = Inches(0.04)
            tf = cell.text_frame
            tf.word_wrap = True
            p = tf.paragraphs[0]
            p.alignment = PP_ALIGN.CENTER
            r = p.add_run(); r.text = val
            r.font.size = Pt(size); r.font.name = FONT
            r.font.bold = (i == 0) or (j == 0)
            r.font.color.rgb = WHITE if i == 0 else DARK


# =============================================================================
# 6. BUILD THE POSTER
# =============================================================================
def build_poster(R):
    os.makedirs(FIG_DIR, exist_ok=True)
    T = make_texts(R)
    prs = Presentation()
    prs.slide_width, prs.slide_height = Inches(POSTER_W), Inches(POSTER_H)
    slide = prs.slides.add_slide(prs.slide_layouts[6])

    M, G = 0.7, 0.6                                        # outer margin, column gap
    colw = (POSTER_W - 2 * M - 2 * G) / 3
    inner = colw - 0.5

    # ---------- header ----------
    HEAD_H = 4.3
    add_rect(slide, 0, 0, POSTER_W, HEAD_H, fill=NAVY)
    add_text(slide, 1.0, 0.35, POSTER_W - 2.0, 2.3, [TITLE], 66, color=WHITE, align=PP_ALIGN.CENTER,
             anchor=MSO_ANCHOR.MIDDLE, after=0)
    # make title bold
    for r in slide.shapes[-1].text_frame.paragraphs[0].runs:
        r.font.bold = True
    add_text(slide, 1.0, 2.75, POSTER_W - 2.0, 0.6, [AUTHORS], 30, color=WHITE, align=PP_ALIGN.CENTER, after=0)
    add_text(slide, 1.0, 3.4, POSTER_W - 2.0, 0.6, [AFFILIATION], 26, color=WHITE, align=PP_ALIGN.CENTER, after=0)

    # ---------- footer ----------
    FOOT_H = 1.5
    add_rect(slide, 0, POSTER_H - FOOT_H, POSTER_W, FOOT_H, fill=NAVY)
    add_text(slide, 0.8, POSTER_H - FOOT_H + 0.1, 20, FOOT_H - 0.2, [([("Contact", True)], False)] +
             CONTACT, 20, color=WHITE, after=0, anchor=MSO_ANCHOR.MIDDLE)
    add_text(slide, POSTER_W - 15.8, POSTER_H - FOOT_H + 0.1, 15, FOOT_H - 0.2, [FOOTER_RIGHT], 24,
             color=WHITE, align=PP_ALIGN.RIGHT, anchor=MSO_ANCHOR.MIDDLE, after=0)

    top = HEAD_H + 0.5
    bottom = POSTER_H - FOOT_H - 0.5
    avail = bottom - top

    # ---------- figure sizes ----------
    f1 = (inner, 5.2)
    fA = (inner, 4.0)
    fB = (inner, 5.6)
    fC = (inner, 3.5)
    flow_h = 8.2
    tbl_rows, tbl_rh = 5, 0.55

    cap = lambda a, b: [([(a, True), (b, False)], False)]

    # item: ("text", paras, size, after) | ("fig", fn, name, (w,h)) | ("flow",) | ("table",) | ("cap", paras)
    cols = [
        [  # ---- column 1
            ("Abstract", [("text", T["abstract"], BODY_PT, 5)]),
            ("Introduction", [("text", T["intro"], BODY_PT, 6)]),
            ("Data Overview", [("text", T["data_bul"], BODY_PT, 4),
                               ("fig", fig_eda, "fig1_eda.png", f1),
                               ("text", cap(*T["fig1_cap"]), CAP_PT, 2)]),
        ],
        [  # ---- column 2
            ("Methods & Materials", [("text", T["methods"], BODY_PT, 5), ("flow",),
                                     ("text", cap(*T["fig2_cap"]), CAP_PT, 2)]),
            ("Results", [("fig", fig_forecast, "fig3A.png", fA),
                         ("fig", fig_forest, "fig3B.png", fB),
                         ("fig", fig_scatter, "fig3C.png", fC),
                         ("text", cap(*T["fig3_cap"]), CAP_PT, 2),
                         ("table",),
                         ("text", cap(*T["table_cap"]), CAP_PT, 2)]),
        ],
        [  # ---- column 3
            ("Discussion", [("text", T["discussion"], BODY_PT, 6)]),
            ("Conclusion", [("text", T["conclusion"], BODY_PT, 6)]),
            ("Limitation", [("text", T["limitation"], BODY_PT, 6)]),
            ("Reference", [("text", T["refs"], 20, 5)]),
        ],
    ]

    def item_h(it):
        if it[0] == "text":
            return est_h(it[1], it[2], colw - 0.4, it[3])
        if it[0] == "fig":
            return it[3][1] + 0.1
        if it[0] == "flow":
            return flow_h + 0.25
        if it[0] == "table":
            return tbl_rh * tbl_rows + 0.15
        return 0

    for ci, sections in enumerate(cols):
        x = M + ci * (colw + G)
        heights = [HDR_H + 0.25 + sum(item_h(it) for it in items) + 0.15 for _, items in sections]
        total = sum(heights)
        ngap = len(sections) - 1
        slack = avail - total - ngap * 0.4
        if slack < 0:
            print(f"WARNING: column {ci + 1} is too full by {-slack:.1f} in - "
                  f"increase POSTER_H or reduce text/figure sizes.")
            slack = 0
        extra_each = slack / len(sections)          # spread free space inside the boxes
        y = top
        for (title, items), hgt in zip(sections, heights):
            hh = hgt + extra_each
            draw_section(slide, x, y, colw, hh, title)
            cy = y + HDR_H + 0.15
            for it in items:
                if it[0] == "text":
                    ih = item_h(it)
                    add_text(slide, x + 0.2, cy, colw - 0.4, ih, it[1], it[2], after=it[3])
                    cy += ih
                elif it[0] == "fig":
                    path = os.path.join(FIG_DIR, it[2])
                    it[1](R, path, *it[3])
                    slide.shapes.add_picture(path, Inches(x + 0.25), Inches(cy), Inches(it[3][0]), Inches(it[3][1]))
                    cy += item_h(it)
                elif it[0] == "flow":
                    draw_flow(slide, x + 0.25, cy + 0.12, inner, flow_h, R)
                    cy += item_h(it)
                elif it[0] == "table":
                    add_table(slide, x + 0.25, cy, inner, R, row_h=tbl_rh)
                    cy += item_h(it)
            y += hh + 0.4

    prs.save(OUT_PPTX)
    print(f"\nPoster saved: {os.path.abspath(OUT_PPTX)}")


# =============================================================================
# 7. MAIN
# =============================================================================
def main():
    R = run_analysis()
    print("\n=== MODEL COMPARISON ===")
    print(R["comp"].round(2))
    print("\n=== IRR (final NB) ===")
    print(R["irr"].round(4))
    build_poster(R)


if __name__ == "__main__":
    main()

Fitting Poisson ...
Fitting Negative Binomial ...
Fitting ZIP ...
Fitting ZINB ...
Fitting final NB on all data ...

=== MODEL COMPARISON ===
              AIC       BIC   k  Dispersion      MAE     RMSE      dAIC
Model                                                                  
Poisson  34998.71  35055.69  17      208.73  2502.13  7983.51  33207.68
NB        1801.17   1861.50  18        0.97  2792.84  7540.00     10.14
ZIP      30161.76  30225.44  19        4.02  2538.18  8049.30  28370.73
ZINB      1791.03   1858.07  20         NaN  2710.12  7551.18      0.00

=== IRR (final NB) ===
               name                        label  IRR  lo  hi   p
0    C(Months)[T.2]            Month: Feb vs Jan  NaN NaN NaN NaN
1    C(Months)[T.3]            Month: Mar vs Jan  NaN NaN NaN NaN
2    C(Months)[T.4]            Month: Apr vs Jan  NaN NaN NaN NaN
3    C(Months)[T.5]            Month: May vs Jan  NaN NaN NaN NaN
4    C(Months)[T.6]            Month: Jun vs Jan  NaN NaN NaN NaN
5    C

In [4]:
!pip install python-pptx


In [8]:
# -*- coding: utf-8 -*-
"""
Application of Generalized Linear Models (GLMs) for Dengue Forecasting in Bangladesh
-----------------------------------------------------------------------------------
ONE script:  data -> models -> figures -> poster (.pptx)

Install once:
    pip install pandas numpy matplotlib seaborn scipy statsmodels scikit-learn patsy openpyxl python-pptx

Run:
    python dengue_poster.py
Output:
    Dengue_Poster.pptx   (+ all figures in ./poster_figs)
"""
import os
import re
import math
import warnings

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from pptx import Presentation
from pptx.util import Inches, Pt
from pptx.dml.color import RGBColor
from pptx.enum.text import PP_ALIGN, MSO_ANCHOR
from pptx.enum.shapes import MSO_SHAPE
from pptx.oxml.ns import qn
from lxml import etree

warnings.filterwarnings("ignore")

# =============================================================================
# 1. CONFIG  (edit this part)
# =============================================================================
DATA_FILE = r"C:\Users\HP\Desktop\STAT 3103\Dengue data.xlsx"          # put the Excel file next to this script
SHEET = "Data"
OUT_PPTX = "Dengue_Poster_v2.pptx"
FIG_DIR = "poster_figs"

POSTER_W, POSTER_H = 36.0, 50.0         # inches (change if your teacher gives a size)

TITLE = "Application of Generalized Linear Models (GLMs) for Dengue Forecasting in Bangladesh"
AUTHORS = "Your Name¹, Team Member 2¹, Team Member 3¹"            # <-- EDIT
AFFILIATION = "¹Department of Statistics, Begum Rokeya University, Rangpur-5400, Bangladesh"  # <-- EDIT
CONTACT = ["Your Name", "Department of Statistics, Begum Rokeya University, Rangpur, Bangladesh",
           "Email: yourmail@gmail.com   |   Phone: 01XXXXXXXXX"]  # <-- EDIT
FOOTER_RIGHT = "Department of Statistics  •  Course: STAT 3103"  # <-- EDIT
# QR code -> link to your public GitHub repo / Google Drive folder (dataset, Python code, results)
# Make the link PUBLIC ("anyone with the link can view"), otherwise the QR will show nothing.
QR_URL = "https://github.com/YOUR_USERNAME/dengue-glm-poster"          # <-- EDIT
QR_NOTE = "Scan: dataset, Python code & results"

TRAIN_FRACTION = 0.80
NUM = ["T2M", "RH2M", "PRECTOTCORR", "Pd"]
RHS = "trend + C(Months) + T2M_z + RH2M_z + PRECTOTCORR_z + Pd_z"
FORMULA = "Cases ~ " + RHS
MODEL_ORDER = ["Poisson", "NB", "ZIP", "ZINB"]
MONTHS = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
LABELS = {"trend": "Trend", "T2M_z": "Temperature",
          "RH2M_z": "Humidity", "PRECTOTCORR_z": "Rainfall",
          "Pd_z": "Pop. density"}
COV_KEYS = list(LABELS.keys())

# ---- poster look ----
FONT = "Arial"              # body text
FONT_HEAD = "Arial Black"   # title + section headers
TITLE_PT, AUTHOR_PT = 60, 50
FIG_PT = 25                 # font size inside figures
NAVY = RGBColor(0x0B, 0x2A, 0x4A)
MID = RGBColor(0x1F, 0x4E, 0x79)
LIGHT = RGBColor(0xE8, 0xEF, 0xF7)
DARK = RGBColor(0x11, 0x11, 0x11)
WHITE = RGBColor(0xFF, 0xFF, 0xFF)
MODEL_COL = {"Poisson": "#d95f02", "NB": "#1b9e77", "ZIP": "#7570b3", "ZINB": "#e7298a"}

BODY_PT = 24
CAP_PT = 20
REF_PT = 19               # reference font (can be < 20 pt)
HDR_PT = 34
HDR_H = 1.0
CHAR_W = 0.52     # avg glyph width (em) used for text-height estimation
LINE_H = 1.17

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Liberation Sans", "DejaVu Sans"],
    "font.size": FIG_PT, "axes.labelsize": FIG_PT, "axes.titlesize": FIG_PT,
    "xtick.labelsize": FIG_PT, "ytick.labelsize": FIG_PT, "legend.fontsize": FIG_PT,
    "axes.spines.top": False, "axes.spines.right": False,
})


# =============================================================================
# 2. DATA + MODELS  (same logic as your notebook)
# =============================================================================
def load_data():
    df = pd.read_excel(DATA_FILE, sheet_name=SHEET)
    df.columns = df.columns.str.strip()
    required = ["Cases", "Year", "Months"] + NUM
    missing = set(required) - set(df.columns)
    if missing:
        raise ValueError(f"Missing columns: {missing}")
    df["Date"] = pd.to_datetime(dict(year=df["Year"], month=df["Months"], day=1), errors="coerce")
    df = df.sort_values("Date").reset_index(drop=True)
    df = df.dropna(subset=required + ["Date"]).copy()
    df["Cases"] = df["Cases"].astype(int)
    if df["Date"].duplicated().any():
        raise ValueError("Resolve duplicate months before modeling.")
    if (df["Cases"] < 0).any():
        raise ValueError("Negative count detected.")
    df["trend"] = (df["Date"] - df["Date"].min()).dt.days / 365.25
    return df.reset_index(drop=True)


def add_z(df, ref):
    """z-score continuous predictors (keeps optimisation stable; IRR is per 1 SD)."""
    df = df.copy()
    for c in NUM:
        df[c + "_z"] = (df[c] - ref[c].mean()) / ref[c].std()
    return df


def robust_fit(make_model, methods=("bfgs", "nm"), maxiter=2000):
    """Try several optimisers until one converges."""
    last = None
    for m in methods:
        try:
            model = make_model()
            res = model.fit(method=m, maxiter=maxiter, disp=0)
            if m == "nm":                         # polish Nelder-Mead with BFGS
                try:
                    res2 = make_model().fit(start_params=res.params, method="bfgs",
                                            maxiter=maxiter, disp=0)
                    res = res2
                except Exception:
                    pass
            last = res
            conv = getattr(res, "mle_retvals", {}).get("converged", True)
            if conv:
                return res
        except Exception as e:
            print(f"   (optimiser {m} failed: {e})")
    if last is None:
        raise RuntimeError("Model could not be fitted.")
    print("   WARNING: model did not fully converge - interpret with care.")
    return last


def run_analysis():
    import statsmodels.api as sm
    import statsmodels.formula.api as smf
    from patsy import dmatrix, build_design_matrices
    from scipy.stats import nbinom
    from statsmodels.discrete.count_model import ZeroInflatedPoisson, ZeroInflatedNegativeBinomialP
    from statsmodels.stats.outliers_influence import variance_inflation_factor
    from sklearn.metrics import mean_absolute_error, mean_squared_error

    data = load_data()
    n = len(data)
    split = int(n * TRAIN_FRACTION)
    train, test = data.iloc[:split].copy(), data.iloc[split:].copy()
    tz, sz, fz = add_z(train, train), add_z(test, train), add_z(data, data)

    # ---------- descriptive / diagnostics ----------
    mean, var = data["Cases"].mean(), data["Cases"].var()
    zeros = int((data["Cases"] == 0).sum())
    Xv = sm.add_constant(data[NUM])
    vif = [variance_inflation_factor(Xv.values, i) for i in range(1, Xv.shape[1])]
    corr_trend_pd = float(np.corrcoef(data["trend"], data["Pd"])[0, 1])

    # ---------- fit the four models on TRAIN ----------
    print("Fitting Poisson ...")
    pois = smf.glm(FORMULA, tz, family=sm.families.Poisson()).fit()
    print("Fitting Negative Binomial ...")
    nb = robust_fit(lambda: smf.negativebinomial(FORMULA, tz), methods=("newton", "bfgs", "nm"))

    X_tr = dmatrix("1 + " + RHS, tz, return_type="dataframe")
    X_te = build_design_matrices([X_tr.design_info], sz, return_type="dataframe")[0]
    Z_tr = dmatrix("1 + trend", tz, return_type="dataframe")
    Z_te = build_design_matrices([Z_tr.design_info], sz, return_type="dataframe")[0]
    y_tr = tz["Cases"]

    print("Fitting ZIP ...")
    zip_m = robust_fit(lambda: ZeroInflatedPoisson(endog=y_tr, exog=X_tr, exog_infl=Z_tr,
                                                   inflation="logit"))
    print("Fitting ZINB ...")
    zinb_m = robust_fit(lambda: ZeroInflatedNegativeBinomialP(endog=y_tr, exog=X_tr, exog_infl=Z_tr,
                                                              inflation="logit", p=2))
    models = {"Poisson": pois, "NB": nb, "ZIP": zip_m, "ZINB": zinb_m}

    # ---------- predictions on TEST ----------
    preds = {
        "Poisson": pois.predict(sz),
        "NB": nb.predict(sz),
        "ZIP": zip_m.predict(exog=X_te, exog_infl=Z_te),
        "ZINB": zinb_m.predict(exog=X_te, exog_infl=Z_te),
    }
    y_test = test["Cases"].values.astype(float)
    for k in preds:
        p = np.nan_to_num(np.asarray(preds[k], dtype=float).ravel(), nan=0.0, posinf=1e10, neginf=0.0)
        preds[k] = np.clip(p, 0, 1e12)

    # ---------- Pearson dispersion for each model (train) ----------
    ytr = y_tr.values.astype(float)

    def pearson(name):
        try:
            m = models[name]
            k = len(m.params)
            if name == "Poisson":
                mu = np.asarray(m.predict(tz)); mean_, v = mu, mu
            elif name == "NB":
                mu = np.asarray(m.predict(tz)); a = float(np.asarray(m.params)[-1])
                mean_, v = mu, mu + a * mu ** 2
            else:
                mu = np.asarray(m.predict(exog=X_tr, exog_infl=Z_tr, which="mean-main"))
                pm = np.asarray(m.predict(exog=X_tr, exog_infl=Z_tr, which="prob-main"))
                pi = 1 - pm
                mean_ = pm * mu
                if name == "ZIP":
                    v = pm * mu * (1 + pi * mu)
                else:
                    a = float(np.asarray(m.params)[-1])
                    v = pm * mu * (1 + a * mu + pi * mu)
            return float(np.sum((ytr - mean_) ** 2 / v) / (len(ytr) - k))
        except Exception:
            return np.nan

    rows = []
    for name in MODEL_ORDER:
        m = models[name]
        k = len(m.params)
        aic = -2 * m.llf + 2 * k
        bic = -2 * m.llf + k * np.log(len(train))
        pr = preds[name]
        rows.append({"Model": name, "AIC": aic, "BIC": bic, "k": k,
                     "Dispersion": pearson(name),
                     "MAE": mean_absolute_error(y_test, pr),
                     "RMSE": float(np.sqrt(mean_squared_error(y_test, pr)))})
    comp = pd.DataFrame(rows).set_index("Model")
    comp["dAIC"] = comp["AIC"] - comp["AIC"].min()

    # ---------- expected zeros (train) ----------
    mu_p = np.asarray(pois.predict(tz))
    a_nb = float(np.asarray(nb.params)[-1])
    mu_n = np.asarray(nb.predict(tz))
    size = 1.0 / a_nb
    exp0_p = float(np.exp(-mu_p).sum())
    exp0_n = float(nbinom.pmf(0, size, size / (size + mu_n)).sum())
    obs0_tr = int((train["Cases"] == 0).sum())

    # ---------- final NB on all data -> IRR ----------
    print("Fitting final NB on all data ...")
    final = robust_fit(lambda: smf.negativebinomial(FORMULA, fz), methods=("newton", "bfgs", "nm"))
    params, ci, pv = final.params, final.conf_int(), final.pvalues
    recs = []
    for name in params.index:
        if name in ("Intercept", "alpha"):
            continue
        if name in LABELS:
            lab = LABELS[name]
        else:
            mm = re.search(r"\[T\.(\d+)\]", name)
            lab = f"Month: {MONTHS[int(mm.group(1)) - 1]} vs Jan" if mm else name
        recs.append({"name": name, "label": lab, "IRR": float(np.exp(params[name])),
                     "lo": float(np.exp(ci.loc[name, 0])), "hi": float(np.exp(ci.loc[name, 1])),
                     "p": float(pv[name])})
    irr = pd.DataFrame(recs)

    return build_result_dict(data, train, test, comp, preds, irr, mean, var, zeros, vif,
                             corr_trend_pd, exp0_p, exp0_n, obs0_tr)


def build_result_dict(data, train, test, comp, preds, irr, mean, var, zeros, vif,
                      corr_trend_pd, exp0_p, exp0_n, obs0_tr):
    fmt = lambda d: d.strftime("%b %Y")
    mm = data.groupby("Months")["Cases"].mean()
    tmax = test.loc[test["Cases"].idxmax()]
    return dict(
        data=data, train=train, test=test, comp=comp, preds=preds, irr=irr,
        n=len(data), n_train=len(train), n_test=len(test),
        period=f"{fmt(data['Date'].min())} – {fmt(data['Date'].max())}",
        train_period=f"{fmt(train['Date'].min())} – {fmt(train['Date'].max())}",
        test_period=f"{fmt(test['Date'].min())} – {fmt(test['Date'].max())}",
        mean=mean, var=var, var_mean=var / mean, zeros=zeros, zero_pct=100 * zeros / len(data),
        vif_max=max(vif), corr_trend_pd=corr_trend_pd,
        exp0_p=exp0_p, exp0_n=exp0_n, obs0_tr=obs0_tr,
        best_fit=comp["AIC"].idxmin(), best_bic=comp["BIC"].idxmin(),
        best_fc=comp["MAE"].idxmin(), best_rmse=comp["RMSE"].idxmin(),
        peak_month=MONTHS[int(mm.idxmax()) - 1], peak_month_mean=float(mm.max()),
        test_max=int(tmax["Cases"]), test_max_date=fmt(tmax["Date"]),
        total_cases=int(data["Cases"].sum()),
        year_max=int(data.groupby("Year")["Cases"].sum().idxmax()),
        year_max_cases=int(data.groupby("Year")["Cases"].sum().max()),
    )


# =============================================================================
# 3. FIGURES  (drawn at the exact size they will have on the poster)
# =============================================================================
def _style(ax):
    ax.grid(alpha=0.25)


def fig_eda(R, path, w, h):
    import matplotlib.dates as mdates
    from matplotlib.ticker import FuncFormatter
    d, tr = R["data"], R["train"]
    fig = plt.figure(figsize=(w, h))
    gs = fig.add_gridspec(1, 2, width_ratios=[1.45, 1], wspace=0.42)
    ax = fig.add_subplot(gs[0])
    ax.plot(d["Date"], d["Cases"], color="#0B2A4A", lw=2)
    ax.axvspan(tr["Date"].iloc[-1], d["Date"].iloc[-1], color="#f4a261", alpha=0.35)
    ax.set_title("(a) Monthly cases", fontweight="bold", loc="left")
    ax.set_ylabel("Cases")
    ax.xaxis.set_major_locator(mdates.YearLocator(10))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v / 1000:.0f}k"))
    _style(ax)
    ax2 = fig.add_subplot(gs[1])
    mm = d.groupby("Months")["Cases"].mean()
    ax2.bar(range(1, 13), mm.values, color="#1F4E79")
    ax2.set_xticks([1, 6, 12])
    ax2.set_xlabel("Month")
    ax2.set_title("(b) Mean by month", fontweight="bold", loc="left")
    ax2.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v / 1000:.0f}k"))
    _style(ax2)
    fig.subplots_adjust(left=0.1, right=0.99, bottom=0.2, top=0.86)
    fig.savefig(path, dpi=200); plt.close(fig)


def fig_forecast(R, path, w, h):
    import matplotlib.dates as mdates
    te = R["test"]
    fig, ax = plt.subplots(figsize=(w, h))
    ax.plot(te["Date"], te["Cases"], color="black", lw=3, marker="o", ms=5, label="Observed", zorder=5)
    for k in MODEL_ORDER:
        ax.plot(te["Date"], R["preds"][k], color=MODEL_COL[k], lw=2, label=k)
    ax.set_yscale("symlog", linthresh=10)
    ax.set_ylabel("Cases")
    ax.xaxis.set_major_locator(mdates.YearLocator(1))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.set_title("A  Observed vs predicted", fontweight="bold", loc="left", pad=62)
    ax.legend(ncol=5, frameon=False, loc="lower left", bbox_to_anchor=(-0.02, 1.0),
              handlelength=1.0, columnspacing=0.7, handletextpad=0.3)
    _style(ax)
    fig.subplots_adjust(left=0.13, right=0.99, bottom=0.13, top=0.76)
    fig.savefig(path, dpi=200); plt.close(fig)


def fig_forest(R, path, w, h):
    from matplotlib.ticker import LogLocator
    irr = R["irr"][R["irr"]["name"].isin(COV_KEYS)].iloc[::-1].reset_index(drop=True)
    fig, ax = plt.subplots(figsize=(w, h))
    y = np.arange(len(irr))
    lo = np.clip(irr["lo"], 1e-3, None); hi = np.clip(irr["hi"], None, 1e6)
    col = ["#c0392b" if p < 0.05 else "#7f8c8d" for p in irr["p"]]
    for yi, l, u, c in zip(y, lo, hi, col):
        ax.plot([l, u], [yi, yi], color=c, lw=3)
    ax.scatter(irr["IRR"], y, c=col, s=110, zorder=3)
    ax.axvline(1, color="black", ls="--", lw=1.5)
    ax.set_xscale("log")
    ax.xaxis.set_major_locator(LogLocator(base=10, numticks=4))
    ax.set_yticks(y); ax.set_yticklabels(irr["label"])
    ax.set_ylim(-0.6, len(irr) - 0.4)
    ax.set_xlabel("IRR (95% CI)")
    ax.set_title("B  Adjusted IRR (final NB)", fontweight="bold", loc="left")
    _style(ax)
    fig.subplots_adjust(left=0.27, right=0.97, bottom=0.2, top=0.86)
    fig.savefig(path, dpi=200); plt.close(fig)


def fig_scatter(R, path, w, h):
    y = R["test"]["Cases"].values.astype(float)
    fig, axes = plt.subplots(2, 2, figsize=(w, h), sharex=True, sharey=True)
    lim = [1, max(y.max(), 10) * 3]
    for ax, k in zip(axes.ravel(), MODEL_ORDER):
        p = np.clip(R["preds"][k], 0, 1e9)
        ax.scatter(y + 1, p + 1, s=45, color=MODEL_COL[k], alpha=0.8)
        ax.plot(lim, lim, color="red", lw=1.5)
        ax.set_xscale("log"); ax.set_yscale("log")
        ax.set_xlim(lim); ax.set_ylim(lim)
        ax.set_xticks([1, 1e2, 1e4]); ax.set_yticks([1, 1e2, 1e4])
        ax.minorticks_off()
        ax.set_title(k, fontweight="bold")
        _style(ax)
    for ax in axes[1]:
        ax.set_xlabel("Observed + 1")
    for ax in axes[:, 0]:
        ax.set_ylabel("Predicted + 1")
    fig.suptitle("C  Predicted vs observed (log scale)", x=0.01, ha="left", fontweight="bold")
    fig.subplots_adjust(left=0.17, right=0.98, bottom=0.11, top=0.88, wspace=0.12, hspace=0.38)
    fig.savefig(path, dpi=200); plt.close(fig)


def make_qr(url, path):
    try:
        import qrcode
    except ImportError:
        print("WARNING: QR code skipped - run  pip install qrcode[pil]  and run again.")
        return None
    if "YOUR_" in url:
        print("WARNING: QR_URL is still the placeholder link - edit QR_URL (public GitHub/Drive link).")
    q = qrcode.QRCode(error_correction=qrcode.constants.ERROR_CORRECT_M, box_size=12, border=2)
    q.add_data(url); q.make(fit=True)
    q.make_image(fill_color="black", back_color="white").save(path)
    return path


# =============================================================================
# 4. POSTER TEXT (numbers come from the real results)
# =============================================================================
def num(x, d=0):
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return "–"
    if abs(x) >= 1e7:
        e = int(math.floor(math.log10(abs(x))))
        sup = str(e).translate(str.maketrans("0123456789-", "⁰¹²³⁴⁵⁶⁷⁸⁹⁻"))
        return f"{x / 10 ** e:.1f}×10{sup}"
    return f"{x:,.{d}f}"


def significant(R):
    irr = R["irr"].set_index("name")
    out = []
    for k in COV_KEYS:
        if k in irr.index and irr.loc[k, "p"] < 0.05:
            r = irr.loc[k]
            out.append((LABELS[k], r["IRR"], r["p"]))
    return out


def sig_text(R):
    s = significant(R)
    if not s:
        return "no covariate reached p < 0.05 in the final NB model"
    return "; ".join(f"{n} (IRR = {i:.2f})" for n, i, _ in s)


def make_texts(R):
    c = R["comp"]
    bf, bc = R["best_fit"], R["best_fc"]
    T = {}
    T["abstract"] = [
        ([("Background", True)], False),
        ("Dengue is a mosquito-borne viral disease and a recurring public-health problem in Bangladesh. "
         "Reliable forecasting of monthly cases, together with knowledge of the factors that drive them, "
         "supports early warning and resource planning.", False),
        ([("Methods", True)], False),
        (f"Monthly dengue cases ({R['period']}, n = {R['n']}) with temperature, humidity, rainfall and "
         f"population density were analysed. Data sources: DGHS Bangladesh (dengue cases), NASA MERRA-2 "
         f"(climate) and the World Bank (population density). Poisson, Negative Binomial (NB), Zero-Inflated Poisson (ZIP) and "
         f"Zero-Inflated NB (ZINB) models were fitted on the first {R['n_train']} months and evaluated on the "
         f"last {R['n_test']} months ({R['test_period']}).", False),
        ([("Results", True)], False),
        ([("Best-fitting model: ", True), (f"{bf} (AIC = {num(c.loc[bf, 'AIC'], 1)}, BIC = {num(c.loc[R['best_bic'], 'BIC'], 1)}).", False)], False),
        ([("Best forecasting model: ", True), (f"{bc} (MAE = {num(c.loc[bc, 'MAE'], 0)}, RMSE = {num(c.loc[R['best_rmse'], 'RMSE'], 0)}).", False)], False),
        ([("Significant factors: ", True), (sig_text(R) + ".", False)], False),
        ([("Seasonality: ", True), (f"cases peak in {R['peak_month']} on average.", False)], False),
        ([("Conclusion", True)], False),
        (f"Dengue counts are strongly over-dispersed (variance/mean = {R['var_mean']:,.0f}) and zero-heavy "
         f"({R['zero_pct']:.1f}% zero months), so count GLMs that handle these features are essential for "
         f"dengue forecasting in Bangladesh.", False),
    ]
    T["intro"] = [
        "Dengue is a mosquito-borne viral infection transmitted mainly by Aedes aegypti and Aedes albopictus. "
        "It is a major public-health problem in tropical and sub-tropical countries (WHO, 2024; Bhatt et al., 2013), "
        "and Bangladesh experiences recurrent outbreaks, mostly in the rainy season.",
        "Monthly dengue counts are non-negative integers, strongly right-skewed, contain many zero months and "
        "show extra-Poisson variation. Ordinary linear regression is therefore unsuitable; Generalized Linear "
        "Models (GLMs; Nelder & Wedderburn, 1972; McCullagh & Nelder, 1989) for counts – Poisson, Negative "
        "Binomial (NB; Hilbe, 2011), Zero-Inflated Poisson (ZIP; Lambert, 1992) and Zero-Inflated NB (ZINB) – "
        "are more appropriate (Cameron & Trivedi, 2013).",
        "Temperature, humidity, rainfall and population density affect mosquito breeding and virus "
        "transmission, so they are natural predictors for an early-warning and forecasting model.",
        ([("Objectives", True)], False),
        ("To forecast monthly dengue cases in Bangladesh using Poisson, NB, ZIP and ZINB models.", "num"),
        ("To measure the association of climate, population density, time trend and seasonality with dengue "
         "incidence through incidence rate ratios (IRR).", "num"),
        ("To select the best-fitting and best-forecasting model using AIC, BIC, MAE and RMSE.", "num"),
    ]
    T["data_bul"] = [
        (f"{R['n']} monthly observations ({R['period']}); {R['total_cases']:,} dengue cases in total.", True),
        (f"Mean = {R['mean']:,.1f} and variance = {R['var']:,.0f} cases/month (variance/mean = {R['var_mean']:,.0f}).", True),
        (f"{R['zeros']} zero-case months ({R['zero_pct']:.1f}%).", True),
        (f"Highest yearly total: {R['year_max']} ({R['year_max_cases']:,} cases).", True),
    ]
    T["fig1_cap"] = ("Figure 1. ", "(a) Monthly dengue cases in Bangladesh (shaded = test period). "
                                   "(b) Average cases by calendar month (1 = Jan, 12 = Dec).")
    T["methods"] = [
        ([("Data: ", True), ("monthly dengue cases (DGHS, Bangladesh), NASA MERRA-2 climate (T2M, RH2M, PRECTOTCORR; "
                             "Gelaro et al., 2017) and World Bank population density (Pd).", False)], True),
        ([("Model: ", True), ("log(μ) = β₀ + β₁·trend + Σ βₘ·Month + β·(T2M, RH2M, PRECTOTCORR, Pd); "
                              "continuous predictors are standardised, IRR = exp(β). Models were compared by "
                              "AIC (Akaike, 1974) and BIC (Schwarz, 1978); fitted with statsmodels (Seabold & "
                              "Perktold, 2010).", False)], True),
        ([("Checks: ", True), (f"variance/mean, Pearson χ²/df and observed vs expected zeros; "
                               f"max VIF = {R['vif_max']:.2f} for the four covariates (O'Brien, 2007).", False)], True),
    ]
    T["fig2_cap"] = ("Figure 2. ", "Study design – from data collection to model selection and interpretation.")
    T["fig3_cap"] = ("Figure 3. ", "A) Observed vs predicted cases (test set); B) adjusted IRR from the final NB model "
                                   "on all data (trend per year, other variables per 1 SD); C) predicted vs "
                                   "observed cases on log scale.")
    T["table_cap"] = ("Table 1. ", "Model selection (training set; χ²/df = Pearson dispersion) and forecast accuracy (test set).")

    sig = significant(R)
    zero_line = (f"Observed zeros in the training data = {R['obs0_tr']}; expected by Poisson = {R['exp0_p']:.1f}, "
                 f"by NB = {R['exp0_n']:.1f}.")
    T["discussion"] = [
        ([(f"{bf} was the best-fitting model", True),
          (f" (lowest AIC = {num(c.loc[bf, 'AIC'], 1)}); BIC favoured {R['best_bic']}.", False)], True),
        ([("Over-dispersion: ", True),
          (f"variance/mean = {R['var_mean']:,.0f}; the Poisson model has Pearson χ²/df = "
           f"{num(c.loc['Poisson', 'Dispersion'], 1)}, so its equal-mean-variance assumption fails (Cameron & Trivedi, 2013).", False)], True),
        ([("Zeros: ", True), (zero_line + " Zero-inflated models target this feature (Lambert, 1992).", False)], True),
        ([(f"{bc} gave the most accurate forecasts", True),
          (f" (MAE = {num(c.loc[bc, 'MAE'], 0)}). The test period contains the largest outbreak "
           f"({R['test_max']:,} cases in {R['test_max_date']}), which makes forecasting difficult.", False)], True),
        ([("Associations: ", True), (sig_text(R) + ".", False)], True),
        ([("Seasonality: ", True), (f"dengue peaks in {R['peak_month']} (mean {R['peak_month_mean']:,.0f} cases), "
                                    f"confirming a strong seasonal pattern.", False)], True),
    ]
    T["conclusion"] = [
        (f"The {bf} model fitted the dengue data best, while {bc} produced the lowest forecast error "
         f"(MAE = {num(c.loc[bc, 'MAE'], 0)}).", True),
        ("Poisson regression is not adequate because of extreme over-dispersion; NB-based models are preferred.", True),
        (f"Time trend, seasonality and climate/population variables explain part of the variation "
         f"({'significant: ' + sig_text(R) if sig else 'no covariate significant'}).", True),
        ("GLM-based forecasts can support early warning, vector-control planning and hospital preparedness.", True),
    ]
    T["limitation"] = [
        ("National monthly data hide district-level and within-month variation.", True),
        (f"Population density and the linear trend are strongly correlated (r = {R['corr_trend_pd']:.2f}), "
         f"so their separate effects are hard to identify.", True),
        ("The test period includes an exceptional outbreak and the COVID-19 period, which may distort forecasts.", True),
        ("No lagged climate effects, spatial effects or vector (Aedes index) data were used.", True),
    ]
    T["refs"] = [
        "Akaike, H. (1974). A new look at the statistical model identification. IEEE Trans. Automat. Control 19(6): 716–723.",
        "Bhatt, S. et al. (2013). The global distribution and burden of dengue. Nature 496: 504–507.",
        "Cameron, A. C. & Trivedi, P. K. (2013). Regression Analysis of Count Data, 2nd ed. Cambridge University Press.",
        "DGHS Bangladesh. Dengue case data: https://doi.org/10.1371/journal.pntd.0010130",
        "Gelaro, R. et al. (2017). The Modern-Era Retrospective Analysis for Research and Applications, Version 2 (MERRA-2). J. Climate 30(14): 5419–5454.",
        "Hilbe, J. M. (2011). Negative Binomial Regression, 2nd ed. Cambridge University Press.",
        "Lambert, D. (1992). Zero-inflated Poisson regression, with an application to defects in manufacturing. Technometrics 34(1): 1–14.",
        "McCullagh, P. & Nelder, J. A. (1989). Generalized Linear Models, 2nd ed. Chapman & Hall.",
        "Nelder, J. A. & Wedderburn, R. W. M. (1972). Generalized linear models. J. R. Stat. Soc. A 135(3): 370–384.",
        "O'Brien, R. M. (2007). A caution regarding rules of thumb for variance inflation factors. Qual. Quant. 41: 673–690.",
        "Schwarz, G. (1978). Estimating the dimension of a model. Ann. Stat. 6(2): 461–464.",
        "Seabold, S. & Perktold, J. (2010). Statsmodels: econometric and statistical modeling with Python. Proc. 9th Python in Science Conf.: 92–96.",
        "World Bank. World Development Indicators: population density. https://data.worldbank.org/indicator/EN.POP.DNST",
        "World Health Organization (2024). Dengue and severe dengue [Fact sheet]. https://www.who.int/news-room/fact-sheets/detail/dengue-and-severe-dengue",
    ]
    return T


# =============================================================================
# 5. PPTX HELPERS
# =============================================================================
def norm(spec):
    """paragraph spec -> (runs, bullet)"""
    bullet = False
    if isinstance(spec, tuple):
        runs, bullet = spec
    else:
        runs = spec
    if isinstance(runs, str):
        runs = [(runs, False)]
    return runs, bullet


def est_h(paras, size, width, after=6, margin=0.12):
    inner = width - 2 * margin
    total = 0.0
    for spec in paras:
        runs, bullet = norm(spec)
        text = "".join(t for t, _ in runs)
        w = inner - (size * 1.5 / 72 if bullet else 0)
        cpl = max(1, int(w * 72 / (size * CHAR_W)))
        lines = 0
        for seg in text.split("\n"):
            cur, nl = 0, 1
            for word in seg.split(" "):
                L = len(word) + (1 if cur else 0)
                if cur + L > cpl:
                    nl += 1; cur = len(word)
                else:
                    cur += L
            lines += nl
        total += lines * size * LINE_H / 72 + after / 72
    return total + 0.1


def set_bullet(p, size, kind=True):
    pPr = p._p.get_or_add_pPr()
    ind = int(Pt(size) * 1.5)
    pPr.set("marL", str(ind)); pPr.set("indent", str(-ind))
    f = etree.SubElement(pPr, qn("a:buFont")); f.set("typeface", "Arial")
    if kind == "num":
        b = etree.SubElement(pPr, qn("a:buAutoNum")); b.set("type", "arabicPeriod")
    else:
        b = etree.SubElement(pPr, qn("a:buChar")); b.set("char", "▪")


def add_text(slide, x, y, w, h, paras, size, color=DARK, align=PP_ALIGN.LEFT,
             anchor=MSO_ANCHOR.TOP, after=6, margin=0.12, font=None):
    tb = slide.shapes.add_textbox(Inches(x), Inches(y), Inches(w), Inches(h))
    tf = tb.text_frame
    tf.word_wrap = True
    tf.margin_left = tf.margin_right = Inches(margin)
    tf.margin_top = tf.margin_bottom = Inches(0.04)
    tf.vertical_anchor = anchor
    first = True
    for spec in paras:
        runs, bullet = norm(spec)
        p = tf.paragraphs[0] if first else tf.add_paragraph()
        first = False
        p.alignment = align
        p.space_after = Pt(after)
        for t, b in runs:
            r = p.add_run(); r.text = t
            r.font.size = Pt(size); r.font.bold = b; r.font.italic = False
            r.font.name = font or FONT; r.font.color.rgb = color
        if bullet:
            set_bullet(p, size, bullet)
    return tb


def add_rect(slide, x, y, w, h, fill=None, line=None, lw=2.0, shape=MSO_SHAPE.RECTANGLE):
    s = slide.shapes.add_shape(shape, Inches(x), Inches(y), Inches(w), Inches(h))
    s.shadow.inherit = False
    if fill is None:
        s.fill.background()
    else:
        s.fill.solid(); s.fill.fore_color.rgb = fill
    if line is None:
        s.line.fill.background()
    else:
        s.line.color.rgb = line; s.line.width = Pt(lw)
    return s


def shape_text(s, text, size, color=WHITE, bold=True, align=PP_ALIGN.CENTER, font=None):
    tf = s.text_frame
    tf.word_wrap = True
    tf.margin_left = tf.margin_right = Inches(0.06)
    tf.margin_top = tf.margin_bottom = Inches(0.02)
    tf.vertical_anchor = MSO_ANCHOR.MIDDLE
    lines = text.split("\n")
    for i, ln in enumerate(lines):
        p = tf.paragraphs[0] if i == 0 else tf.add_paragraph()
        p.alignment = align
        r = p.add_run(); r.text = ln
        r.font.size = Pt(size); r.font.name = font or FONT; r.font.color.rgb = color
        r.font.italic = False
        r.font.bold = bold if i == 0 else False


def draw_section(slide, x, y, w, h, title):
    add_rect(slide, x, y, w, h, fill=WHITE, line=NAVY, lw=2.5)
    hdr = add_rect(slide, x, y, w, HDR_H, fill=NAVY)
    shape_text(hdr, title, HDR_PT, bold=False, font=FONT_HEAD)


def draw_flow(slide, x, y, w, h, R):
    stages = [
        ("Data collection", [f"Dengue cases\nDGHS, n = {R['n']}", "Climate\nT2M, RH2M, rain",
                             "Demography\nPop. density"]),
        ("Data preprocessing", ["Quality checks • date index • trend, month • VIF"]),
        ("Time-based split", [f"Training (80%)\n{R['train_period']}", f"Test (20%)\n{R['test_period']}"]),
        ("Diagnostics", ["Over-dispersion\nVar/Mean, χ²/df", "Excess zeros\nobs. vs expected"]),
        ("GLM fitting (training set)", ["Poisson", "Negative\nBinomial", "ZIP", "ZINB"]),
        ("Evaluation (test set)", ["AIC / BIC", "Pearson χ²/df", "MAE", "RMSE"]),
        ("Best model", ["Final NB model on all data → IRR interpretation"]),
    ]
    n = len(stages)
    arrow_h = 0.2
    band_h = (h - (n - 1) * arrow_h) / n
    cy = y
    for i, (title, boxes) in enumerate(stages):
        band = add_rect(slide, x, cy, w, band_h, fill=LIGHT, line=MID, lw=1.25, shape=MSO_SHAPE.ROUNDED_RECTANGLE)
        band.adjustments[0] = 0.12
        add_text(slide, x, cy + 0.03, w, 0.45, [([(title, True)], False)], 21, color=NAVY,
                 align=PP_ALIGN.CENTER, after=0, margin=0.05)
        k = len(boxes)
        gap = 0.12
        bw = (w - 0.3 - (k - 1) * gap) / k
        bh = band_h - 0.58
        for j, t in enumerate(boxes):
            b = add_rect(slide, x + 0.15 + j * (bw + gap), cy + 0.52, bw, bh, fill=MID,
                         shape=MSO_SHAPE.ROUNDED_RECTANGLE)
            b.adjustments[0] = 0.15
            shape_text(b, t, 19)
        cy += band_h
        if i < n - 1:
            a = add_rect(slide, x + w / 2 - 0.2, cy + 0.01, 0.4, arrow_h - 0.02, fill=NAVY,
                         shape=MSO_SHAPE.DOWN_ARROW)
            cy += arrow_h


def add_table(slide, x, y, w, R, size=20, row_h=0.65):
    c = R["comp"]
    head = ["Model", "AIC", "BIC", "ΔAIC", "χ²/df", "MAE", "RMSE"]
    rows = [head]
    for k in MODEL_ORDER:
        r = c.loc[k]
        rows.append([k, num(r["AIC"], 1), num(r["BIC"], 1), num(r["dAIC"], 2),
                     num(r["Dispersion"], 2), num(r["MAE"], 0), num(r["RMSE"], 0)])
    shp = slide.shapes.add_table(len(rows), len(head), Inches(x), Inches(y), Inches(w),
                                 Inches(row_h * len(rows)))
    tbl = shp.table
    widths = [1.5, 1.7, 1.7, 1.3, 1.7, 1.5, 1.7]
    sc = w / sum(widths)
    for i, wd in enumerate(widths):
        tbl.columns[i].width = Inches(wd * sc)
    for i, row in enumerate(rows):
        tbl.rows[i].height = Inches(row_h)
        for j, val in enumerate(row):
            cell = tbl.cell(i, j)
            cell.fill.solid()
            cell.fill.fore_color.rgb = NAVY if i == 0 else (LIGHT if i % 2 == 0 else WHITE)
            cell.vertical_anchor = MSO_ANCHOR.MIDDLE
            cell.margin_left = cell.margin_right = Inches(0.04)
            tf = cell.text_frame
            tf.word_wrap = True
            p = tf.paragraphs[0]
            p.alignment = PP_ALIGN.CENTER
            r = p.add_run(); r.text = val
            r.font.size = Pt(size); r.font.name = FONT; r.font.italic = False
            r.font.bold = (i == 0) or (j == 0)
            r.font.color.rgb = WHITE if i == 0 else DARK


# =============================================================================
# 6. BUILD THE POSTER
# =============================================================================
def build_poster(R):
    os.makedirs(FIG_DIR, exist_ok=True)
    T = make_texts(R)
    prs = Presentation()
    prs.slide_width, prs.slide_height = Inches(POSTER_W), Inches(POSTER_H)
    slide = prs.slides.add_slide(prs.slide_layouts[6])

    M, G = 0.7, 0.6                                        # outer margin, column gap
    colw = (POSTER_W - 2 * M - 2 * G) / 3
    inner = colw - 0.5

    # ---------- header (title 60 pt : authors 50 pt) ----------
    HEAD_H = 6.0
    add_rect(slide, 0, 0, POSTER_W, HEAD_H, fill=NAVY)
    add_text(slide, 1.0, 0.25, POSTER_W - 2.0, 3.1, [([(TITLE, False)], False)], TITLE_PT, color=WHITE,
             align=PP_ALIGN.CENTER, anchor=MSO_ANCHOR.MIDDLE, after=0, font=FONT_HEAD)
    add_text(slide, 1.0, 3.5, POSTER_W - 2.0, 1.0, [AUTHORS], AUTHOR_PT, color=WHITE,
             align=PP_ALIGN.CENTER, anchor=MSO_ANCHOR.MIDDLE, after=0)
    add_text(slide, 1.0, 4.7, POSTER_W - 2.0, 0.8, [AFFILIATION], 30, color=WHITE,
             align=PP_ALIGN.CENTER, anchor=MSO_ANCHOR.MIDDLE, after=0)

    # ---------- footer + QR ----------
    FOOT_H = 2.0
    fy = POSTER_H - FOOT_H
    add_rect(slide, 0, fy, POSTER_W, FOOT_H, fill=NAVY)
    add_text(slide, 0.8, fy + 0.1, 17, FOOT_H - 0.2, [([("Contact", True)], False)] + CONTACT, 22,
             color=WHITE, after=0, anchor=MSO_ANCHOR.MIDDLE)
    qr_size = FOOT_H - 0.3
    qr_x = POSTER_W - 0.8 - qr_size
    add_text(slide, qr_x - 13.3, fy + 0.1, 13, FOOT_H - 0.2, [FOOTER_RIGHT, QR_NOTE], 24,
             color=WHITE, align=PP_ALIGN.RIGHT, anchor=MSO_ANCHOR.MIDDLE, after=2)
    add_rect(slide, qr_x - 0.08, fy + 0.07, qr_size + 0.16, qr_size + 0.16, fill=WHITE)
    qr_path = make_qr(QR_URL, os.path.join(FIG_DIR, "qr.png"))
    if qr_path:
        slide.shapes.add_picture(qr_path, Inches(qr_x), Inches(fy + 0.15), Inches(qr_size), Inches(qr_size))

    top = HEAD_H + 0.5
    bottom = POSTER_H - FOOT_H - 0.5
    avail = bottom - top

    # ---------- figure sizes ----------
    f1 = (inner, 5.6)
    fA = (inner, 6.0)
    fB = (inner, 5.0)
    fC = (inner, 8.0)
    flow_h = 11.6
    tbl_rows, tbl_rh = 5, 0.65

    cap = lambda a, b: [([(a, True), (b, False)], False)]

    # item: ("text", paras, size, after) | ("fig", fn, name, (w,h)) | ("flow",) | ("table",) | ("cap", paras)
    cols = [
        [  # ---- column 1
            ("Abstract", [("text", T["abstract"], BODY_PT, 5)]),
            ("Introduction", [("text", T["intro"], BODY_PT, 6)]),
            ("Data Overview", [("text", T["data_bul"], BODY_PT, 4),
                               ("fig", fig_eda, "fig1_eda.png", f1),
                               ("text", cap(*T["fig1_cap"]), CAP_PT, 2)]),
        ],
        [  # ---- column 2
            ("Methods & Materials", [("text", T["methods"], BODY_PT, 5), ("flow",),
                                     ("text", cap(*T["fig2_cap"]), CAP_PT, 2)]),
            ("Results", [("fig", fig_forecast, "fig3A.png", fA),
                         ("fig", fig_forest, "fig3B.png", fB),
                         ("fig", fig_scatter, "fig3C.png", fC),
                         ("text", cap(*T["fig3_cap"]), CAP_PT, 2),
                         ]),
        ],
        [  # ---- column 3
            ("Model Comparison", [("table",), ("text", cap(*T["table_cap"]), CAP_PT, 2)]),
            ("Discussion", [("text", T["discussion"], BODY_PT, 6)]),
            ("Conclusion", [("text", T["conclusion"], BODY_PT, 6)]),
            ("Limitation", [("text", T["limitation"], BODY_PT, 6)]),
            ("Reference", [("text", T["refs"], REF_PT, 4)]),
        ],
    ]

    def item_h(it):
        if it[0] == "text":
            return est_h(it[1], it[2], colw - 0.4, it[3])
        if it[0] == "fig":
            return it[3][1] + 0.1
        if it[0] == "flow":
            return flow_h + 0.25
        if it[0] == "table":
            return tbl_rh * tbl_rows + 0.15
        return 0

    for ci, sections in enumerate(cols):
        x = M + ci * (colw + G)
        heights = [HDR_H + 0.25 + sum(item_h(it) for it in items) + 0.15 for _, items in sections]
        total = sum(heights)
        ngap = len(sections) - 1
        slack = avail - total - ngap * 0.4
        if slack < 0:
            print(f"WARNING: column {ci + 1} is too full by {-slack:.1f} in - "
                  f"increase POSTER_H or reduce text/figure sizes.")
            slack = 0
        extra_each = slack / len(sections)          # spread free space inside the boxes
        y = top
        for (title, items), hgt in zip(sections, heights):
            hh = hgt + extra_each
            draw_section(slide, x, y, colw, hh, title)
            cy = y + HDR_H + 0.15
            for it in items:
                if it[0] == "text":
                    ih = item_h(it)
                    add_text(slide, x + 0.2, cy, colw - 0.4, ih, it[1], it[2], after=it[3])
                    cy += ih
                elif it[0] == "fig":
                    path = os.path.join(FIG_DIR, it[2])
                    it[1](R, path, *it[3])
                    slide.shapes.add_picture(path, Inches(x + 0.25), Inches(cy), Inches(it[3][0]), Inches(it[3][1]))
                    cy += item_h(it)
                elif it[0] == "flow":
                    draw_flow(slide, x + 0.25, cy + 0.12, inner, flow_h, R)
                    cy += item_h(it)
                elif it[0] == "table":
                    add_table(slide, x + 0.25, cy, inner, R, row_h=tbl_rh)
                    cy += item_h(it)
            y += hh + 0.4

    prs.save(OUT_PPTX)
    print(f"\nPoster saved: {os.path.abspath(OUT_PPTX)}")


# =============================================================================
# 7. MAIN
# =============================================================================
def main():
    R = run_analysis()
    print("\n=== MODEL COMPARISON ===")
    print(R["comp"].round(2))
    print("\n=== IRR (final NB) ===")
    print(R["irr"].round(4))
    build_poster(R)


if __name__ == "__main__":
    main()

Fitting Poisson ...
Fitting Negative Binomial ...
Fitting ZIP ...
Fitting ZINB ...
Fitting final NB on all data ...

=== MODEL COMPARISON ===
              AIC       BIC   k  Dispersion      MAE     RMSE      dAIC
Model                                                                  
Poisson  34998.71  35055.69  17      208.73  2502.13  7983.51  33207.68
NB        1801.17   1861.50  18        0.97  2792.84  7540.00     10.14
ZIP      30161.76  30225.44  19        4.02  2538.18  8049.30  28370.73
ZINB      1791.03   1858.07  20         NaN  2710.12  7551.18      0.00

=== IRR (final NB) ===
               name              label  IRR  lo  hi   p
0    C(Months)[T.2]  Month: Feb vs Jan  NaN NaN NaN NaN
1    C(Months)[T.3]  Month: Mar vs Jan  NaN NaN NaN NaN
2    C(Months)[T.4]  Month: Apr vs Jan  NaN NaN NaN NaN
3    C(Months)[T.5]  Month: May vs Jan  NaN NaN NaN NaN
4    C(Months)[T.6]  Month: Jun vs Jan  NaN NaN NaN NaN
5    C(Months)[T.7]  Month: Jul vs Jan  NaN NaN NaN NaN
6    C(Mon